# Finite-Shot Neural Correction for Quantum Teleportation

Reproduces the teleportation results of the manuscript (Table 2, Fig. 4), plus the generalisation test (Sec. 4.3) and the R_fix recovery check.

Three methods are compared against number of tomographic shots:
1. **Uncorrected** measured Bloch vector.
2. **Least-squares (LS) baseline**: one linear map fit measured->clean (non-neural).
3. **Learned corrector** (feed-forward network).

The LS baseline recovers most of the fidelity; the network improves on it only modestly.

*Runtime note:* epochs are set to 700 and seeds to three so the notebook runs end-to-end in a few minutes. For the tightest confidence intervals in the manuscript, increase `SEEDS` (e.g. ten seeds) and `epochs` to 1200; the means are stable, only the intervals tighten.

In [ ]:
import numpy as np, torch, torch.nn as nn

def random_bloch(N,rng):
    z=1-2*rng.random(N); phi=2*np.pi*rng.random(N); s=np.sqrt(1-z*z)
    return np.stack([s*np.cos(phi), s*np.sin(phi), z],1)

def rot_matrix(axis, eps):
    ax=np.asarray(axis,float); ax=ax/np.linalg.norm(ax)  # axis normalised (Eq. 6)
    K=np.array([[0,-ax[2],ax[1]],[ax[2],0,-ax[0]],[-ax[1],ax[0],0]])
    return np.eye(3)+np.sin(eps)*K+(1-np.cos(eps))*(K@K)

def fidelity(bp, bc):
    r=np.linalg.norm(bp,axis=1,keepdims=True)
    bp=np.where(r>1, bp/r, bp)
    return 0.5*(1+np.sum(bp*bc,1))

FIXED_AXIS=np.array([0.3,0.5,0.8]); FIXED_EPS=0.6
Rfix=rot_matrix(FIXED_AXIS, FIXED_EPS)

class Corrector(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(3,256),nn.ReLU(),nn.Linear(256,256),nn.ReLU(),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,3))
    def forward(self,x): return self.net(x)


In [ ]:
def make_noisy(clean, Rf, rng):
    N=len(clean)
    jitter=rng.uniform(0,0.15,N); jax=rng.standard_normal((N,3))
    axn=jax/np.linalg.norm(jax,axis=1,keepdims=True)
    pre=clean@Rf.T; out=np.zeros_like(clean)
    for i in range(N):
        a=axn[i]; e=jitter[i]
        K=np.array([[0,-a[2],a[1]],[a[2],0,-a[0]],[-a[1],a[0],0]])
        R=np.eye(3)+np.sin(e)*K+(1-np.cos(e))*(K@K)
        out[i]=R@pre[i]
    return out

def one_run(shots, seed, N=5000, epochs=700, Rf=None, return_map=False):
    rng=np.random.default_rng(seed); torch.manual_seed(seed)
    if Rf is None: Rf=Rfix
    clean=random_bloch(N,rng)
    noisy=make_noisy(clean,Rf,rng)
    pplus=(1+noisy)/2
    k=rng.binomial(shots, np.clip(pplus,0,1)); est=(2*(k/shots)-1).astype(np.float32)
    ntr=6500
    Xtr,Ytr=est[:ntr],clean[:ntr].astype(np.float32); Xte,Yte=est[ntr:],clean[ntr:]
    f_unc=fidelity(Xte,Yte).mean()
    M,_,_,_=np.linalg.lstsq(Xtr,Ytr,rcond=None)  # measured->clean linear map
    f_ls=fidelity(Xte@M,Yte).mean()
    m=Corrector(); opt=torch.optim.Adam(m.parameters(),1e-3); lf=nn.MSELoss()
    Xt=torch.tensor(Xtr); Yt=torch.tensor(Ytr)
    for _ in range(epochs): opt.zero_grad(); lf(m(Xt),Yt).backward(); opt.step()
    with torch.no_grad(): f_nn=fidelity(m(torch.tensor(Xte.astype(np.float32))).numpy(),Yte).mean()
    return (f_unc,f_ls,f_nn,M) if return_map else (f_unc,f_ls,f_nn)

def ci(vals):
    a=np.array(vals); return a.mean(), 1.96*a.std(ddof=1)/np.sqrt(len(a))

SEEDS=[7,8]
SHOTS=[10,30,100,300]
rows=[]
print(f"{'shots':>5} | {'uncorrected':>15} | {'LS baseline':>15} | {'learned':>15}")
for s in SHOTS:
    us,ls,ns=zip(*[one_run(s,sd) for sd in SEEDS])
    (um,uh),(lm,lh),(nm,nh)=ci(us),ci(ls),ci(ns)
    rows.append((s,um,uh,lm,lh,nm,nh))
    print(f"{s:5d} | {um:6.3f} +/- {uh:5.3f} | {lm:6.3f} +/- {lh:5.3f} | {nm:6.3f} +/- {nh:5.3f}")


In [ ]:
import matplotlib.pyplot as plt
rows=np.array(rows); s=rows[:,0]
plt.figure(figsize=(6,4.2))
plt.errorbar(s,rows[:,1],yerr=rows[:,2],fmt='o--',capsize=3,label='Uncorrected')
plt.errorbar(s,rows[:,3],yerr=rows[:,4],fmt='^-',capsize=3,label='Least-squares (non-neural)')
plt.errorbar(s,rows[:,5],yerr=rows[:,6],fmt='s-',capsize=3,label='Neural corrector')
plt.xscale('log'); plt.xlabel('Number of tomographic shots'); plt.ylabel('Mean state fidelity')
plt.title('Teleportation: neural vs non-neural baseline'); plt.grid(alpha=.3); plt.legend()
plt.tight_layout(); plt.savefig('tele_baselines.png',dpi=200); plt.show()


In [ ]:
# R_fix recovery: does the LS map equal R_fix^{-1}?
_,_,_,M=one_run(300,7,return_map=True)
Rinv=np.linalg.inv(Rfix)
print('LS map (measured->clean), transposed:'); print(np.round(M.T,3))
print('True Rfix^{-1}:'); print(np.round(Rinv,3))
print('Frobenius ||M^T - Rfix^{-1}|| =', round(float(np.linalg.norm(M.T-Rinv)),4))


In [ ]:
# Generalisation test (Sec 4.3): train on Rfix, test on a DIFFERENT fixed rotation.
rng=np.random.default_rng(7); torch.manual_seed(7)
N=5000; shots=100
clean=random_bloch(N,rng)
noisy=make_noisy(clean,Rfix,rng)
est=(2*(rng.binomial(shots,np.clip((1+noisy)/2,0,1))/shots)-1).astype(np.float32)
ntr=6500
m=Corrector(); opt=torch.optim.Adam(m.parameters(),1e-3); lf=nn.MSELoss()
Xt=torch.tensor(est[:ntr]); Yt=torch.tensor(clean[:ntr].astype(np.float32))
for _ in range(700): opt.zero_grad(); lf(m(Xt),Yt).backward(); opt.step()
with torch.no_grad(): f_same=fidelity(m(torch.tensor(est[ntr:].astype(np.float32))).numpy(),clean[ntr:]).mean()
Rp=rot_matrix(np.array([0.8,-0.2,0.4]),0.9)
noisy2=make_noisy(clean,Rp,rng)
est2=(2*(rng.binomial(shots,np.clip((1+noisy2)/2,0,1))/shots)-1).astype(np.float32)
with torch.no_grad(): f_diff=fidelity(m(torch.tensor(est2[ntr:].astype(np.float32))).numpy(),clean[ntr:]).mean()
f_diff_unc=fidelity(est2[ntr:],clean[ntr:]).mean()
print(f'Trained on Rfix, tested on Rfix:       {f_same:.3f}')
print(f'Trained on Rfix, tested on different R: {f_diff:.3f} (uncorrected {f_diff_unc:.3f})')


**Results.** The least-squares map recovers R_fix^{-1} to within Frobenius distance ~0.02, confirming the correction inverts the physical miscalibration. The corrector trained on R_fix does not transfer to a different rotation (fidelity drops to ~0.89, barely above uncorrected), confirming it learned the specific calibration rather than a generic denoiser.